# Page-segmentation detector (Lens-style) — Colab training

Trains a **MobileNetV2-UNet** to segment document pages on **SmartDoc 2015 Challenge 1** (CC BY 4.0), then exports an int8 TFLite.

1. `Runtime → Change runtime type → GPU` (a free T4 is enough).
2. Run all cells top to bottom.
3. The last cell downloads `*_dynamic_range_quant.tflite`.

Dataset: <https://zenodo.org/records/1230218> · 1.4 GB · cite the SmartDoc 2015 ICDAR paper.

In [ ]:
# Colab already has torch / torchvision / opencv / numpy / onnx / onnxruntime.
!pip install -q segmentation-models-pytorch albumentations onnx2tf
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


## 1. Download + extract the dataset (1.4 GB)

In [ ]:
!rm -rf /content/smartdoc /content/ready /content/testDataset.tar.gz
!mkdir -p /content/smartdoc
!curl -L -o /content/testDataset.tar.gz "https://zenodo.org/api/records/1230218/files/testDataset.tar.gz/content"
!tar -xzf /content/testDataset.tar.gz -C /content/smartdoc
!echo '--- ground truth ---'; find /content/smartdoc -name '*.gt.xml' | head -3
!echo '--- clips ---'; find /content/smartdoc -name '*.avi' | head -3
!du -sh /content/smartdoc


## 2. The pipeline (written to disk, then run)

In [ ]:
%%writefile prepare_dataset.py
#!/usr/bin/env python
"""Extract frames + page masks from SmartDoc 2015 Challenge 1.

The dataset (https://zenodo.org/records/1230218, CC BY 4.0) unpacks into
``backgroundXX/<doc><NNN>.avi`` clips plus ``backgroundXX_gt/<doc><NNN>.gt.xml``
per-frame quadrilaterals.

Output layout (what train.py consumes):

    <out>/images/<clip>_<frame>.jpg   RGB/BGR frame, full resolution
    <out>/masks/<clip>_<frame>.png    255 = page, 0 = background
    <out>/train.txt, val.txt          one "images/...jpg" path per line

Usage:
    python prepare_dataset.py --data C:/data/smartdoc --out C:/data/smartdoc_ready
"""
import argparse
import os
import glob
import xml.etree.ElementTree as ET

import cv2
import numpy as np


def parse_gt(xml_path):
    """Return {frame_index (1-based): [tl, tr, br, bl]} for accepted frames."""
    root = ET.parse(xml_path).getroot()
    out = {}
    for fr in root.iter("frame"):
        if fr.get("rejected") == "true":
            continue
        pts = {
            p.get("name"): (float(p.get("x")), float(p.get("y")))
            for p in fr.iter("point")
        }
        if all(k in pts for k in ("tl", "tr", "br", "bl")):
            out[int(fr.get("index"))] = [pts["tl"], pts["tr"], pts["br"], pts["bl"]]
    return out


def find_video(gt_path, data_root):
    """Locate the .avi/.mp4 that a .gt.xml belongs to.

    Clip names repeat across backgrounds (``background01/datasheet001.avi`` and
    ``background02/datasheet001.avi``), so the match is scoped to the same
    ``backgroundXX`` folder the ground truth came from.
    """
    stem = os.path.basename(gt_path).replace(".gt.xml", "")
    bg = os.path.basename(os.path.dirname(gt_path))
    if bg.endswith("_gt"):
        bg = bg[:-3]
    for ext in (".avi", ".mp4"):
        hits = glob.glob(os.path.join(data_root, "**", bg, stem + ext), recursive=True)
        if hits:
            return hits[0]
    return None


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data", required=True, help="unpacked SmartDoc 2015 root")
    ap.add_argument("--out", required=True, help="output dir for images/masks")
    ap.add_argument("--stride", type=int, default=2,
                    help="keep every Nth accepted frame (default 2)")
    ap.add_argument("--val-background", default="background03",
                    help="background held out for validation (default background03)")
    args = ap.parse_args()

    img_dir = os.path.join(args.out, "images")
    msk_dir = os.path.join(args.out, "masks")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(msk_dir, exist_ok=True)

    gts = sorted(glob.glob(os.path.join(args.data, "**", "*.gt.xml"), recursive=True))
    if not gts:
        raise SystemExit(f"no *.gt.xml under {args.data} — extract the dataset first")

    def background(p):
        name = os.path.basename(os.path.dirname(p))
        return name[:-3] if name.endswith("_gt") else name

    backgrounds = sorted({background(p) for p in gts})
    val_bg = args.val_background if args.val_background in backgrounds else backgrounds[-1]
    print(f"backgrounds: {backgrounds}  -> validation: {val_bg}")

    train, val = [], []
    kept = 0
    for gi, gt_path in enumerate(gts):
        bg = background(gt_path)
        is_val = bg == val_bg
        frames = parse_gt(gt_path)
        if not frames:
            continue
        video = find_video(gt_path, args.data)
        if video is None:
            print(f"  !! no video for {gt_path}")
            continue

        cap = cv2.VideoCapture(video)
        if not cap.isOpened():
            print(f"  !! cannot open {video}")
            continue
        clip = os.path.splitext(os.path.basename(video))[0]
        idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            gt_index = idx + 1
            quad = frames.get(gt_index)
            if quad is not None and idx % args.stride == 0:
                name = f"{clip}_{gt_index:05d}"
                h, w = frame.shape[:2]
                mask = np.zeros((h, w), np.uint8)
                poly = np.round(np.array(quad, np.float32)).astype(np.int32)
                cv2.fillPoly(mask, [poly], 255)
                cv2.imwrite(os.path.join(img_dir, name + ".jpg"), frame,
                            [cv2.IMWRITE_JPEG_QUALITY, 90])
                cv2.imwrite(os.path.join(msk_dir, name + ".png"), mask)
                (val if is_val else train).append(f"images/{name}.jpg")
                kept += 1
            idx += 1
        cap.release()

    with open(os.path.join(args.out, "train.txt"), "w") as f:
        f.write("\n".join(train) + "\n")
    with open(os.path.join(args.out, "val.txt"), "w") as f:
        f.write("\n".join(val) + "\n")

    print(f"wrote {kept} frames: train {len(train)}, val {len(val)}")
    print(f"val background: {args.val_background}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
#!/usr/bin/env python
"""Train a MobileNetV2-UNet page segmenter on SmartDoc 2015.

Same architecture family as Lens's detector (MobileNetV2 encoder + decoder,
256x256 RGB -> 256x256 mask), sized for a phone.

    python train.py --data C:/data/smartdoc_ready --epochs 40 --size 256

Saves ``best.pth`` (highest validation Dice) next to the run.
"""
import argparse
import os

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


class DocDataset(Dataset):
    def __init__(self, root, list_file, transform):
        self.root = root
        with open(list_file) as f:
            self.items = [line.strip() for line in f if line.strip()]
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        rel = self.items[i]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.root, rel)), cv2.COLOR_BGR2RGB)
        mask_path = os.path.join(self.root, rel.replace("images/", "masks/").replace(".jpg", ".png"))
        mask = cv2.imread(mask_path, 0)
        aug = self.transform(image=img, mask=mask)
        x = aug["image"]
        y = (aug["mask"] > 127).float().unsqueeze(0)
        return x, y


def transforms(size):
    train = A.Compose([
        A.LongestMaxSize(max_size=int(size * 1.5)),
        A.PadIfNeeded(min_height=int(size * 1.5), min_width=int(size * 1.5), border_mode=0),
        A.RandomCrop(height=size, width=size),
        A.Perspective(scale=(0.02, 0.08), p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.RandomBrightnessContrast(p=0.5),
        A.RandomGamma(p=0.5),
        A.GaussianBlur(p=0.2),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    val = A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    return train, val


def dice_loss(logits, target):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    union = p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + 1) / (union + 1)).mean()


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    dices, ious = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        p = torch.sigmoid(model(x))
        pred = (p > 0.5).float()
        inter = (pred * y).sum(dim=(1, 2, 3))
        union = pred.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3))
        dice = (2 * inter + 1) / (union + 1)
        iou = (inter + 1) / (pred.sum(dim=(1, 2, 3)) + y.sum(dim=(1, 2, 3)) - inter + 1)
        dices.append(dice.mean().item())
        ious.append(iou.mean().item())
    return float(np.mean(dices)), float(np.mean(ious))


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data", required=True)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--size", type=int, default=256)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--encoder", default="mobilenet_v2")
    ap.add_argument("--out", default="best.pth")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")

    train_tf, val_tf = transforms(args.size)
    train_ds = DocDataset(args.data, os.path.join(args.data, "train.txt"), train_tf)
    val_ds = DocDataset(args.data, os.path.join(args.data, "val.txt"), val_tf)
    train_ld = DataLoader(train_ds, batch_size=args.batch, shuffle=True,
                          num_workers=args.workers, pin_memory=True, drop_last=True)
    val_ld = DataLoader(val_ds, batch_size=args.batch, shuffle=False,
                        num_workers=args.workers, pin_memory=True)

    model = smp.Unet(encoder_name=args.encoder, encoder_weights="imagenet",
                     in_channels=3, classes=1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    best = -1.0
    for epoch in range(1, args.epochs + 1):
        model.train()
        running = 0.0
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = F.binary_cross_entropy_with_logits(logits, y) + dice_loss(logits, y)
            loss.backward()
            opt.step()
            running += loss.item() * x.size(0)
        sched.step()
        dice, iou = evaluate(model, val_ld, device)
        print(f"epoch {epoch:03d}  loss {running / len(train_ds):.4f}  "
              f"val dice {dice:.4f}  val iou {iou:.4f}")
        if dice > best:
            best = dice
            torch.save(model.state_dict(), args.out)
            print(f"  saved {args.out} (dice {dice:.4f})")
    print(f"best val dice {best:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_tflite.py
#!/usr/bin/env python
"""Export a trained MobileNetV2-UNet to int8 TFLite for the app.

    python export_tflite.py --weights best.pth --size 256

Produces ``tflite_out/<name>_dynamic_range_quant.tflite`` (weights int8,
float activations) which runs on the app's existing ``tflite_flutter`` runtime
with no calibration set. For full-integer (activations too) add a calibration
folder and use ``-oiqt`` instead of ``-odrqt``.
"""
import argparse
import subprocess
import sys

import torch
import segmentation_models_pytorch as smp


def main():
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--weights", default="best.pth")
    ap.add_argument("--encoder", default="mobilenet_v2")
    ap.add_argument("--size", type=int, default=256)
    ap.add_argument("--onnx", default="doc_segmenter.onnx")
    ap.add_argument("--out", default="tflite_out")
    args = ap.parse_args()

    model = smp.Unet(encoder_name=args.encoder, encoder_weights=None,
                     in_channels=3, classes=1)
    model.load_state_dict(torch.load(args.weights, map_location="cpu"))
    model.eval()

    dummy = torch.randn(1, 3, args.size, args.size)
    torch.onnx.export(
        model, dummy, args.onnx,
        input_names=["input"], output_names=["mask"],
        opset_version=16,
        dynamic_axes={"input": {0: "batch"}, "mask": {0: "batch"}},
    )
    print(f"onnx: {args.onnx}")

    cmd = [sys.executable, "-m", "onnx2tf", "-i", args.onnx, "-o", args.out, "-odrqt"]
    print("running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print(f"done: {args.out}  (pick the *_dynamic_range_quant.tflite)")


if __name__ == "__main__":
    main()


## 3. Frames + masks

In [ ]:
!python prepare_dataset.py --data /content/smartdoc --out /content/ready --stride 2
!echo; wc -l /content/ready/train.txt /content/ready/val.txt


## 4. Train (~a few hours on a T4)

In [ ]:
!python train.py --data /content/ready --epochs 40 --size 256


## 5. Export to int8 TFLite

In [ ]:
!python export_tflite.py --weights best.pth --size 256
!ls -la tflite_out/*.tflite


## 6. Download the model

In [ ]:
import glob
from google.colab import files
path = glob.glob('tflite_out/*_dynamic_range_quant.tflite')[0]
print('downloading', path)
files.download(path)
